In [2]:
# Data extraction
import polars as pl
import glob # pattern matching

# Visualisation
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import warnings

warnings.filterwarnings('ignore')

In [ ]:
files = sorted(glob.glob('./data/EV__*.parquet'))
lf = pl.scan_parquet(files)
lf.head(10).collect()

In [4]:
EIN = '[72_1]S7-Schweißmaschine - Schweißen EIN (M215_0)' # rename target column

''' aggregation segment
    0 --> 1 1 --> 0 
    now & prev                
'''

df_aug = (
    lf.select('Time', EIN)
    .drop_nulls()
    .with_columns(
        seq_id=(pl.col(EIN) & ~pl.col(EIN).shift(fill_value=False)).cum_sum()
    )
    .group_by('seq_id')
    .agg(pl.col(EIN).first(), start_timer=pl.col('Time').min(),
         end_timer=pl.col('Time').max(), n_samples=pl.len())
    .sort('seq_id')
    .collect(engine='streaming')
)

In [ ]:
SEQ = 2
s = df_aug.filter(pl.col('seq_id') == SEQ).row(0, named=True)
start_time, end_time = s['start_timer'], s['end_timer']

cols = [c for c in lf.collect_schema().names() if c != 'Time']

df_seq = (
    lf
    .filter(pl.col('Time').is_between(start_time, end_time))
    .select('Time', *cols)
    .gather_every(100)
    .collect(engine='streaming')
)

for i in range(0, len(cols), 8):
    batch = cols[i:i+8]
    fig = make_subplots(rows=len(batch), cols=1, shared_xaxes=True,
                        vertical_spacing=0.03, subplot_titles=batch)

    for r, c in enumerate(batch, start=1):
        fig.add_trace(go.Scattergl(
            x=df_seq['Time'], y=df_seq[c].cast(pl.Float32),
            mode='lines', line=dict(width=1.5, color='darkorchid'),
            name=c, showlegend=False,
            hovertemplate='%{y:.3g}<extra></extra>',
        ), row=r, col=1)


    # looks b
    fig.update_layout(height=220 * len(batch), template='plotly_white',
                      plot_bgcolor='#e5e5e5',
                      paper_bgcolor='#f5f5f5',
                      hovermode='x unified', margin=dict(l=60, r=20, t=40, b=30))
    fig.update_annotations(font_size=11, x=0, xanchor='left')
    fig.update_xaxes(showspikes=True, spikemode='across', spikethickness=1, spikecolor='#555555',
                     gridcolor='white')
    fig.update_yaxes(gridcolor='white')
    fig.write_image(f'./Images/seq{SEQ}_part{i // 8}.png', width=1500, scale=2)
    fig.show()